In [1]:
import pandas as pd
import sqlite3

CLEAN_PATH = r"C:\Users\2539990\Downloads\healthcare_cleaned.csv"

# ─────────────────────────────────────────────────────────────────────────────
# LOAD CLEANED DATA INTO SQLite IN-MEMORY DATABASE
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("  STEP 1 │ Load Cleaned Data into SQLite")
print("=" * 65)

df = pd.read_csv(CLEAN_PATH)
conn = sqlite3.connect(":memory:")
df.to_sql("patients", conn, index=False, if_exists="replace")

row_count = pd.read_sql_query("SELECT COUNT(*) AS n FROM patients", conn).iloc[0, 0]
print(f"\n  Table   : patients")
print(f"  Rows    : {row_count:,}")
print(f"  Columns : {list(df.columns)}")

def run_query(title, sql, conn):
    print(f"\n{'─' * 65}")
    print(f"  QUERY: {title}")
    print(f"{'─' * 65}")
    # Print the SQL with indentation
    for line in sql.strip().splitlines():
        print(f"  {line}")
    print(f"\n  ── Result ──")
    result = pd.read_sql_query(sql, conn)
    print(result.to_string(index=False))
    return result

  STEP 1 │ Load Cleaned Data into SQLite

  Table   : patients
  Rows    : 997
  Columns : ['Patient Name', 'Age', 'Age_Group', 'Gender', 'Condition', 'Medication', 'On_Medication', 'Visit Date', 'Visit_Year', 'Visit_Month', 'BP_Systolic', 'BP_Diastolic', 'BP_Category', 'Cholesterol', 'High_Cholesterol', 'Email', 'Phone Number']


In [2]:
#  QUERIES
print("\n" + "=" * 65)
print("  STEP 2 │ SQL Queries")
print("=" * 65)

results = {}

# ── Q1: Patient counts & averages per condition ───────────────────────────────
results["q1"] = run_query(
    "Patient Summary by Condition",
    """
    SELECT
        Condition,
        COUNT(*)                        AS total_patients,
        ROUND(AVG(Age), 1)              AS avg_age,
        ROUND(MIN(Age), 0)              AS min_age,
        ROUND(MAX(Age), 0)              AS max_age,
        ROUND(AVG(Cholesterol), 1)      AS avg_cholesterol,
        ROUND(AVG(BP_Systolic), 1)      AS avg_systolic_bp
    FROM patients
    GROUP BY Condition
    ORDER BY total_patients DESC;
    """,
    conn
)

# ── Q2: Gender breakdown ─────────────────────────────────────────────────────
results["q2"] = run_query(
    "Average Blood Pressure by Gender",
    """
    SELECT
        Gender,
        COUNT(*)                        AS n,
        ROUND(AVG(BP_Systolic), 1)      AS avg_systolic,
        ROUND(AVG(BP_Diastolic), 1)     AS avg_diastolic,
        ROUND(AVG(Cholesterol), 1)      AS avg_cholesterol
    FROM patients
    GROUP BY Gender
    ORDER BY n DESC;
    """,
    conn
)


  STEP 2 │ SQL Queries

─────────────────────────────────────────────────────────────────
  QUERY: Patient Summary by Condition
─────────────────────────────────────────────────────────────────
  SELECT
          Condition,
          COUNT(*)                        AS total_patients,
          ROUND(AVG(Age), 1)              AS avg_age,
          ROUND(MIN(Age), 0)              AS min_age,
          ROUND(MAX(Age), 0)              AS max_age,
          ROUND(AVG(Cholesterol), 1)      AS avg_cholesterol,
          ROUND(AVG(BP_Systolic), 1)      AS avg_systolic_bp
      FROM patients
      GROUP BY Condition
      ORDER BY total_patients DESC;

  ── Result ──
    Condition  total_patients  avg_age  min_age  max_age  avg_cholesterol  avg_systolic_bp
       Asthma             415     44.8     25.0     70.0            187.4            126.3
Heart Disease             207     45.6     25.0     70.0            186.3            125.4
     Diabetes             204     45.8     25.0     70.0   

In [3]:
# ── Q3: Medication usage per condition ───────────────────────────────────────
results["q3"] = run_query(
    "Medication Frequency by Condition",
    """
    SELECT
        Condition,
        Medication,
        COUNT(*)                                    AS count,
        ROUND(COUNT(*) * 100.0 /
            SUM(COUNT(*)) OVER (PARTITION BY Condition), 1) AS pct_within_condition
    FROM patients
    GROUP BY Condition, Medication
    ORDER BY Condition, count DESC;
    """,
    conn
)

# ── Q4: Yearly visit volume ───────────────────────────────────────────────────
results["q4"] = run_query(
    "Yearly & Monthly Visit Volume",
    """
    SELECT
        Visit_Year,
        Visit_Month,
        COUNT(*) AS visits
    FROM patients
    GROUP BY Visit_Year, Visit_Month
    ORDER BY Visit_Year, Visit_Month;
    """,
    conn
)

# ── Q5: High-risk patients ────────────────────────────────────────────────────
results["q5"] = run_query(
    "High-Risk Patients (Systolic BP ≥ 140 AND Cholesterol > 200)",
    """
    SELECT
        "Patient Name",
        Age,
        Gender,
        Condition,
        BP_Systolic,
        BP_Diastolic,
        Cholesterol,
        Medication
    FROM patients
    WHERE BP_Systolic >= 140
      AND Cholesterol > 200
    ORDER BY Cholesterol DESC, BP_Systolic DESC
    LIMIT 15;
    """,
    conn
)


─────────────────────────────────────────────────────────────────
  QUERY: Medication Frequency by Condition
─────────────────────────────────────────────────────────────────
  SELECT
          Condition,
          Medication,
          COUNT(*)                                    AS count,
          ROUND(COUNT(*) * 100.0 /
              SUM(COUNT(*)) OVER (PARTITION BY Condition), 1) AS pct_within_condition
      FROM patients
      GROUP BY Condition, Medication
      ORDER BY Condition, count DESC;

  ── Result ──
    Condition   Medication  count  pct_within_condition
       Asthma          NaN     88                  21.2
       Asthma Atorvastatin     84                  20.2
       Asthma    Albuterol     81                  19.5
       Asthma   Lisinopril     81                  19.5
       Asthma    Metformin     81                  19.5
     Diabetes    Metformin     47                  23.0
     Diabetes   Lisinopril     41                  20.1
     Diabetes    Albuterol  

In [4]:
# ── Q6: BP category distribution across conditions ───────────────────────────
results["q6"] = run_query(
    "Blood Pressure Category Distribution by Condition",
    """
    SELECT
        Condition,
        BP_Category,
        COUNT(*) AS n,
        ROUND(COUNT(*) * 100.0 /
            SUM(COUNT(*)) OVER (PARTITION BY Condition), 1) AS pct
    FROM patients
    GROUP BY Condition, BP_Category
    ORDER BY Condition, n DESC;
    """,
    conn
)

# ── Q7: Age group risk analysis ───────────────────────────────────────────────
results["q7"] = run_query(
    "Risk Indicators by Age Group",
    """
    SELECT
        Age_Group,
        COUNT(*)                                    AS total,
        SUM(High_Cholesterol)                       AS high_chol_count,
        ROUND(AVG(High_Cholesterol) * 100, 1)       AS pct_high_chol,
        ROUND(AVG(BP_Systolic), 1)                  AS avg_systolic,
        SUM(On_Medication)                          AS on_medication,
        ROUND(AVG(On_Medication) * 100, 1)          AS pct_medicated
    FROM patients
    GROUP BY Age_Group
    ORDER BY Age_Group;
    """,
    conn
)

# ── Q8: Patients with no medication and high risk ────────────────────────────
results["q8"] = run_query(
    "Unmedicated High-Risk Patients (may need intervention)",
    """
    SELECT
        Condition,
        COUNT(*) AS unmedicated_high_risk
    FROM patients
    WHERE On_Medication = 0
      AND (High_Cholesterol = 1 OR BP_Systolic >= 130)
    GROUP BY Condition
    ORDER BY unmedicated_high_risk DESC;
    """,
    conn)


─────────────────────────────────────────────────────────────────
  QUERY: Blood Pressure Category Distribution by Condition
─────────────────────────────────────────────────────────────────
  SELECT
          Condition,
          BP_Category,
          COUNT(*) AS n,
          ROUND(COUNT(*) * 100.0 /
              SUM(COUNT(*)) OVER (PARTITION BY Condition), 1) AS pct
      FROM patients
      GROUP BY Condition, BP_Category
      ORDER BY Condition, n DESC;

  ── Result ──
    Condition  BP_Category   n  pct
       Asthma       Normal 166 40.0
       Asthma     Elevated 153 36.9
       Asthma High Stage 1  96 23.1
     Diabetes       Normal  82 40.2
     Diabetes     Elevated  68 33.3
     Diabetes High Stage 1  54 26.5
Heart Disease       Normal  90 43.5
Heart Disease     Elevated  74 35.7
Heart Disease High Stage 1  43 20.8
 Hypertension       Normal  69 40.4
 Hypertension     Elevated  62 36.3
 Hypertension High Stage 1  40 23.4

─────────────────────────────────────────────────

In [5]:
## SUMMARY
print("\n" + "=" * 65)
print("  STEP 3 │ Key Findings Summary")
print("=" * 65)

most_common = results["q1"].iloc[0]
print(f"\n  • Most common condition    : {most_common['Condition']} ({most_common['total_patients']} patients)")

high_risk_count = len(results["q5"])
print(f"  • High-risk patients found : {high_risk_count} (SBP≥140 & Chol>200)")

unmed_total = results["q8"]["unmedicated_high_risk"].sum()
print(f"  • Unmedicated high-risk    : {unmed_total} patients may need clinical review")

conn.close()
print("\n  ✔  SQL analysis complete. Run 03_ml_modeling.py next.")


  STEP 3 │ Key Findings Summary

  • Most common condition    : Asthma (415 patients)
  • High-risk patients found : 15 (SBP≥140 & Chol>200)
  • Unmedicated high-risk    : 125 patients may need clinical review

  ✔  SQL analysis complete. Run 03_ml_modeling.py next.
